# 04 · E6 Fraud in the legitimate training reference (ULB)

What happens to OCSVM and the autoencoder when the "legitimate" data they learn from secretly contains fraud?

For each seed, a reference of 30,000 training transactions is built with 0, 30, 60, 150 or 300 frauds in it (0 to 1.0%). OCSVM and the AE are fitted on that reference. IF is fitted on all training rows, as in the main protocol, because it never relied on a clean reference. Features, validation and test partitions stay the same within a seed, and the threshold is the usual percentile with c = 0.002.

The references are nested: the 60-fraud reference contains the 30 frauds of the 30-fraud one, and the legitimate rows are dropped from the same end of one fixed shuffled list. Differences between levels therefore come from the added fraud, not from drawing new legitimate rows.

> **Rebuilt experiment.** The v5 notebook that produced `training_reference_contamination_by_seed.csv` was not among the uploaded files, so this follows the manuscript's description (fixed-size 30,000 reference, five contamination levels, unchanged test partitions). If you find the original, compare it with the choices written above, especially whether IF was also fitted on the reference.

| | |
|---|---|
| Trains models | yes: 5 seeds × 5 levels × (OCSVM + AE), plus one IF per seed |
| Expected time | roughly 15-20 min on a GPU runtime (each AE trains on 30,000 rows) |
| Writes | `results/E6_reference_contamination/`: `reference_contamination_by_seed.csv`, `reference_contamination_summary.csv` |

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.load_config(ROOT)
env = rg.show_environment()

In [ ]:
import gc
import numpy as np, pandas as pd

EXP = "E6_reference_contamination"
FORCE_RERUN = False
RUN_TRAINING = FORCE_RERUN or not rg.already_done(cfg, EXP, ["reference_contamination_by_seed.csv"])
print("Training will run." if RUN_TRAINING else "E6 results exist - loading them.")

X, y, feature_names = rg.load_ulb(cfg)
W, C = cfg["fusion_weights"], cfg["contamination"]
REF_N = cfg["e6"]["reference_size"]
SHARES = cfg["e6"]["fraud_shares"]
print("Frauds per level:", [int(round(s * REF_N)) for s in SHARES])

In [ ]:
if RUN_TRAINING:
    run = rg.Run(cfg, EXP, "04_E6_reference_contamination.ipynb", "Fraud injected into the OCSVM/AE training reference")
    rows = []
    for seed in cfg["seeds"]:
        tr, va, te = rg.get_split(cfg, "ulb", y, seed)
        y_tr, y_va, y_te = y[tr], y[va], y[te]
        F = rg.prepare_features(X[tr], X[va], X[te], feature_names, seed, cfg)

        if_scores, if_secs, _ = rg.fit_detectors(F, y_tr, seed, cfg, detectors=("IF",))

        rng = np.random.RandomState(seed)
        legit = rng.permutation(np.flatnonzero(y_tr == 0))
        fraud = rng.permutation(np.flatnonzero(y_tr == 1))
        assert len(fraud) >= round(max(SHARES) * REF_N), "not enough training frauds for the largest level"
        assert len(legit) >= REF_N

        for share in SHARES:
            k = int(round(share * REF_N))
            ref = np.concatenate([legit[:REF_N - k], fraud[:k]])
            scores, secs, info = rg.fit_detectors(F, y_tr, seed, cfg, detectors=("OCSVM", "AE"),
                                                  fit_rows={"OCSVM": ref, "AE": ref})
            scores.update(if_scores)
            val_s = rg.weighted_fusion({m: scores[m][1] for m in scores}, W)
            test_s = rg.weighted_fusion({m: scores[m][2] for m in scores}, W)
            thr = rg.percentile_threshold(val_s, C)
            rows.append({"Seed": seed, "Training_Reference_Contamination": share,
                         "Reference_N": len(ref), "Reference_Fraud_N": k, "Threshold_Calibration_C": C,
                         "Train_Time_s": secs["OCSVM"] + secs["AE"] + if_secs["IF"],
                         "AE_Epochs": info["AE_epochs"], **rg.evaluate(y_te, test_s, thr, val_s)})
            r = rows[-1]
            print(f"seed {seed:>4} | {k:>3} frauds | PR-AUC {r['PR_AUC']:.3f} | ROC-AUC {r['ROC_AUC']:.3f} | F1 {r['F1']:.3f}")
        del F
        gc.collect()

    res = pd.DataFrame(rows)
    run.save_csv(res, "reference_contamination_by_seed.csv")
    run.save_csv(rg.mean_sd(res, ["Training_Reference_Contamination"],
                            rg.METRICS + ["TP_per_10k", "FP_per_10k", "Alerts_per_10k"]),
                 "reference_contamination_summary.csv")
    run.finish()

In [ ]:
summary = rg.read_result(cfg, EXP, "reference_contamination_summary.csv")
display(summary[["Training_Reference_Contamination", "PR_AUC_mean", "PR_AUC_sd", "ROC_AUC_mean",
                 "F1_mean", "TP_per_10k_mean", "FP_per_10k_mean"]].round(4))